# 12 - Governance States And The Wider Estate

Metatate's typed answers are honest about uncertainty, and the estate is big
enough to prove it: a deliberately UNGOVERNED legacy table, a monitored custom
masking routine served as review-required, role-gated HR data, PCI-scope
payment instruments, AI-lifecycle rules on an ML feature store,
taxonomy-targeted masking that follows the classification — not a column
list — plus a deliberately CONFLICTED policy pair, the retain/conditional/
log_only decision vocabulary, the free-text scenario front door, and a
second governed schema.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. The honest states

An ungoverned asset is a typed `not_enough_published_state` — never an empty
pass. A custom masking routine the engine cannot map deterministically is a
typed `review_required` — a human must decide.


In [ ]:
ungoverned = client.authorize_use(
    asset("legacy_customer_backup"),
    use="report on the legacy customer backup",
    scenario_key="purpose.allowed_use",
)
print(f"legacy_customer_backup -> {ungoverned['state']} ({ungoverned.get('reason_code')})")
for action in ungoverned.get("next_actions") or []:
    print(f"  next: {action}")


In [ ]:
review = client.authorize_use(
    asset("employees", "full_name"),
    use="display employee names in the people directory",
    scenario_key="masking.display",
)
print(f"employees.full_name masking -> {review['state']} ({review.get('reason_code')})")


## 2. Role-gated access and public-sharing prohibitions

`access.read` on employee records cites BOTH role rules — the deny wins, and
the allow for people-ops roles is right there in the citations.


In [ ]:
hr_read = client.authorize_use(
    asset("employees"),
    use="browse employee records",
    scenario_key="access.read",
)
print_answer(hr_read)
for instruction in hr_read["instructions"]:
    print(f"  cited: {instruction['instruction_key']} -> {instruction['decision']}")


In [ ]:
sharing = client.authorize_use(
    asset("employees"),
    use="publish the org chart externally",
    scenario_key="sharing.public",
)
print(f"sharing.public -> {sharing['decision']}")


## 3. The AI lifecycle, scenario by scenario

Training on RAW ticket text is denied — but training on DERIVED features is
allowed. Retrieval context and embedding storage are permitted; vendor
transfer and fully automated decisioning are not.


In [ ]:
lifecycle = [
    ("raw tickets, ai.training", client.authorize_use(
        asset("support_tickets"),
        use="fine-tune a support assistant on ticket text",
        scenario_key="ai.training",
    )),
    ("features, ai.training", client.authorize_use(
        asset("ml_feature_store"),
        use="train the churn model on derived features",
        scenario_key="ai.training",
    )),
    ("features, ai.retrieval_context", client.authorize_use(
        asset("ml_feature_store"),
        use="feed churn features into agent retrieval context",
        scenario_key="ai.retrieval_context",
    )),
    ("features, ai.embedding_storage", client.authorize_use(
        asset("ml_feature_store"),
        use="index feature vectors in the embedding store",
        scenario_key="ai.embedding_storage",
    )),
    ("features, ai.vendor_transfer", client.authorize_use(
        asset("ml_feature_store"),
        use="share churn features with an external AI vendor",
        scenario_key="ai.vendor_transfer",
    )),
    ("features, ai.automated_decisioning", client.authorize_use(
        asset("ml_feature_store"),
        use="auto-cancel accounts from churn scores",
        scenario_key="ai.automated_decisioning",
    )),
]
for name, answer in lifecycle:
    print(f"{name:40s} -> {answer_label(answer)}")


## 4. Taxonomy-targeted masking: classify once, govern everywhere

One policy targets the taxonomy type `pii.contact.email` — no column lists.
The engine resolved it to every column the catalog classifies as an email,
including the HR table nobody edited the policy for.


In [ ]:
work_email = client.validate_query_context(
    "SELECT work_email FROM employees",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"SELECT work_email -> {work_email['verdict']}")
for finding in work_email["findings"]:
    for instruction in finding["instructions"]:
        paths = ",".join(p["source"] for p in instruction.get("resolution_paths") or [])
        print(f"  {instruction['decision']} via [{paths}] {instruction.get('decision_reason')}")


## 5. Collections: govern the named set, not a table list

The "Customer 360" COLLECTION groups five tables; one policy targets the
collection by id. The answer's citations carry a `collection` resolution
path — add a table to the collection and the policy follows it.


In [ ]:
sharing = client.authorize_use(
    asset("subscriptions"),
    use="share account health summaries with the success team",
    scenario_key="sharing.internal",
)
print(f"sharing.internal on subscriptions -> {sharing['decision']}")
for instruction in sharing["instructions"]:
    paths = ",".join(p["source"] for p in instruction.get("resolution_paths") or [])
    print(f"  cited via [{paths}]: {instruction['provenance']['policy_name']}")


## 6. PCI-scope payment data and intent-less reads


In [ ]:
card = client.validate_query_context(
    "SELECT card_last4 FROM payment_methods",
    scenario_key="purpose.allowed_use",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"card_last4 (analytics intent) -> {card['verdict']} (tokenized column referenced)")

salary = client.validate_query_context(
    "SELECT salary FROM employees",
    default_database="acmecloud_demo",
    default_schema="public",
)
print(f"salary (NO stated intent)    -> {salary['verdict']} (role-gated read applies to any SQL)")


## 7. Governance debt is a typed state, not a coin flip

Two policies deliberately disagree about outreach on
`marketing_prospects`: Growth Marketing permits the exact use the
Privacy Office prohibits, at the same priority, on the same scenario.
The engine refuses to pick a side — the answer is a typed
`review_required(conflicted_published_state)` citing BOTH sources.


In [ ]:
conflict = client.authorize_use(
    asset("marketing_prospects"),
    use="run outreach against the prospect list",
    scenario_key="purpose.allowed_use",
)
print(f"marketing_prospects outreach -> {conflict['state']} ({conflict.get('reason_code')})")
for source in (conflict.get("conflict") or {}).get("sources") or []:
    print(f"  cited: {source['provenance']['policy_name']} -> {source['decision']}")


## 8. The rest of the decision vocabulary

Not every answer is allow/deny. Retention serves an honest `retain`
(with a structured obligation), row-level access serves `conditional`
with a `role_restricted` condition, compliance context serves
`log_only`, and an enforced PCI mask serves `mask_full` with a `mask`
obligation naming the method.


In [ ]:
retention = client.authorize_use(
    asset("subscriptions"),
    use="confirm how long subscription revenue facts must be kept",
    scenario_key="retention.lifecycle",
)
print_answer(retention)


In [ ]:
rows = client.authorize_use(
    asset("employees"),
    use="read employee rows for my region",
    scenario_key="access.row_filter",
)
print_answer(rows)


In [ ]:
gdpr = client.authorize_use(
    asset("employees"),
    use="review the GDPR context for employee records",
    scenario_key="compliance.regulatory",
)
print(f"compliance.regulatory -> {gdpr['decision']} (regulatory context, not a permission)")

pci = client.authorize_use(
    asset("payment_methods", "card_token"),
    use="display stored payment instruments in the support console",
    scenario_key="masking.display",
)
print_answer(pci)


## 9. The free-text front door

No `scenario_key` at all: the server maps the plain-English use to a
canonical scenario deterministically — and when the text is ambiguous,
it refuses to guess with a typed `scenario_unresolved`.


In [ ]:
mapped = client.authorize_use(
    asset("support_tickets"),
    use="fine-tune a model on this data",
)
print(f"free text, no scenario_key -> {mapped['state']} / {mapped.get('decision')} (mapped to {mapped.get('scenario_key')})")

ambiguous = client.authorize_use(
    asset("customers"),
    use="compare ai.training and ai.inference guidance for customer data",
)
print(f"ambiguous free text        -> {ambiguous['state']} ({ambiguous.get('reason_code')})")


## 10. A second schema, same decision layer

The estate spans two schemas: `finance.invoices` and
`finance.revenue_ledger` are governed by their own guardrail policy —
schema-qualified assets answer exactly like `public` ones.


In [ ]:
invoices = client.authorize_use(
    asset("invoices", schema="finance"),
    use="prepare the quarterly revenue recognition report",
    scenario_key="purpose.allowed_use",
)
print(f"finance.invoices reporting            -> {invoices['decision']}")

ledger = client.authorize_use(
    asset("revenue_ledger", schema="finance"),
    use="publish ledger extracts on the public status page",
    scenario_key="sharing.public",
)
print(f"finance.revenue_ledger public sharing -> {ledger['decision']}")
